# TD5 — Mini-agent: code your own PIM enrichment agent (the capstone)

**~1h15 · the fifth and final brick — it recomposes TD1 → TD4 into one agent.**

### Three beats to set the scene

**1. You already used an AI agent.** In **TD4** you registered your PIM tools and asked Claude Desktop a question — and *it* picked which tool to call and *wrote the arguments itself*. That was an agent making **one** decision. You didn't build the agent, though; Claude Desktop was the agent.

**2. What is an AI agent?** An **AI agent** is an LLM running in a **loop** that **reasons → acts (calls a tool) → observes (reads the result)**, deciding each step for itself, until the goal is reached. Contrast it with a plain LLM call: one prompt → one answer, no tools, no follow-through. You'll build it in two moves: first run **one** turn of this loop by hand (reason → act → observe, once), then **close the loop** so the agent repeats the turn until the job is done.

**3. The purpose: code our own agent from scratch.** We rebuild the lecture's **distributor PIM enrichment agent**. A supplier emails us about new products in messy prose; the agent reads it and enriches the catalog on its own — **reusing your own bricks**: the **TD3 RAG** (now the `search_products` tool) and the **TD4 MCP server** (the agent's whole toolbox). The agent categorizes, fetches the schema, grounds in similar products, writes an on-brand entry, and creates the product.

**How to work:** run top to bottom; complete the **5 `# TODO`** cells — **four short steps** that build one reason→act→observe turn (reason · find the tool · act+observe · final answer), then the **`run_agent` loop** that generalizes them (the centerpiece, with a detailed docstring to guide you). Everything downstream — the `enrich_all` driver that runs the agent over the whole supplier email — is **given**. Each TODO ends with a **self-check** so you can confirm your code without any solution. Write your own analysis answers (use Claude Code to *think*, not to write for you).

> ⚠️ **This is the most API-intensive lab.** Unlike every prior TD, the agent makes **many** Haiku calls in a loop (one per reason→act→observe turn, across a Q&A goal and two product enrichments). You need `ANTHROPIC_API_KEY` in a `.env` file at the project root. Everything else is local (embeddings, the in-process MCP server). The loop is bounded by a `max_iters` guard so it can't run away. 🚀

## 1. Setup (all pre-built — your bricks, reassembled)

This whole section is given. It reassembles the course: the **TD3 RAG index** (catalog in ChromaDB, same MiniLM), the **TD4 MCP server** with its tools — plus the two new pieces TD5 needs: a **`create_product` write tool**, and the agent's **skill**. Read it, run it, then build the loop in §3.

Nothing here is the lesson — the lesson is the **loop** you write next. We give you a working toolbox so you can focus on the agent.

In [ ]:
import json
import os
import logging
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
import anthropic

# MCP server + in-memory client transport (real list_tools / call_tool, no subprocess) -- same as TD4.
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session as connect

# The MCP server logs every request at INFO; quiet it so notebook output stays readable.
logging.getLogger("mcp").setLevel(logging.WARNING)

# Shared PIM catalog + taxonomy -- the same dataset used in TD1-TD4.
df = pd.read_csv("../data/products.csv")
df["doc"] = df["name"] + " — " + df["long_description"]
with open("../data/taxonomy.json") as f:
    taxonomy = json.load(f)

print(f"Loaded {len(df)} products across {df['category'].nunique()} leaf categories.")

In [ ]:
# Rebuild the TD3 RAG index: embed the catalog locally with MiniLM and add the vectors to Chroma
# EXPLICITLY -- the same one shared vector space as TD1 -> TD3 -> TD4 -> TD5.
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
if "catalog" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("catalog")
collection = chroma_client.create_collection("catalog")

embeddings = embed_model.encode(df["doc"].tolist(), show_progress_bar=False)
collection.add(
    ids=df["sku"].tolist(),
    embeddings=embeddings.tolist(),
    documents=df["doc"].tolist(),
    metadatas=[{"name": r["name"], "brand": r["brand"], "category": r["category"],
                "price": float(r["price"]), "short_description": r["short_description"],
                "long_description": r["long_description"],
                "attributes": r["attributes"], "extra": "{}"}
               for _, r in df.iterrows()],
)
print(f"Indexed {collection.count()} products into ChromaDB.")

In [ ]:
# Pre-built RAG retriever -- same as TD3/TD4, returning each hit's FULL stored metadata so the
# agent (and any caller) gets the whole product, attributes and all.
def retrieve(query_text, k=3):
    """Return the k catalog products most similar to `query_text`, each as a dict carrying the
    product's FULL stored metadata (sku + everything saved in the DB)."""
    q_vec = embed_model.encode(query_text).tolist()
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    hits = []
    for sku, meta in zip(res["ids"][0], res["metadatas"][0]):
        hit = {"sku": sku, **meta}
        for field in ("attributes", "extra"):   # stored as JSON strings -> parse back to dicts
            if isinstance(hit.get(field), str):
                hit[field] = json.loads(hit[field])
        hits.append(hit)
    return hits

# Smoke test the retriever.
for h in retrieve("noise cancelling headphones", k=3):
    print(f"  - {h['name']}  ({h['category']}, €{h['price']:.0f})")

In [ ]:
# Build the in-memory MCP server with the FOUR tools the agent gets. These are the TD4 tools
# (get_category_tree, get_category_attributes, search_products) PLUS create_product (the write op,
# deferred to TD4's mini-project, defined here for the agent to call).
mcp_server = FastMCP("pim")

def get_category_tree_plain():
    return {
        cat["name"]: [sub["name"] for sub in cat["subcategories"]]
        for cat in taxonomy["categories"]
    }

@mcp_server.tool()
def get_category_tree() -> dict:
    """Return the PIM category tree as {top_category: [leaf_category, ...]}."""
    return get_category_tree_plain()

@mcp_server.tool()
def get_category_attributes(category: str) -> dict:
    """Return the applicable attribute schema for a leaf category, as {attribute: type/values}."""
    for top in taxonomy["categories"]:
        for sub in top["subcategories"]:
            if sub["name"] == category:
                out = {}
                for a in sub["category_attributes"]:
                    if a.get("values"):
                        out[a["name"]] = "enum: " + ", ".join(a["values"])
                    else:
                        out[a["name"]] = a["type"] + (f" ({a['unit']})" if a.get("unit") else "")
                return out
    return {}  # unknown category -> fail gracefully

@mcp_server.tool()
def search_products(query: str, k: int = 3) -> list:
    """Semantic search over the product catalog; returns up to k products most similar to the query."""
    return retrieve(query, k=k)

@mcp_server.tool()
def get_product(sku: str) -> dict:
    """Fetch a single product by its SKU; returns its descriptions, attributes, and extra."""
    got = collection.get(ids=[sku], include=["documents", "metadatas"])
    if not got["ids"]:
        return {"error": f"no product with sku {sku!r}"}
    meta = dict(got["metadatas"][0])
    meta["attributes"] = json.loads(meta.get("attributes", "{}"))
    meta["extra"] = json.loads(meta.get("extra", "{}"))
    return {"sku": sku, "document": got["documents"][0], **meta}

@mcp_server.tool()
def create_product(sku: str, name: str, brand: str, category: str, price: float,
                   short_description: str, long_description: str,
                   attributes: dict, extra: dict | None = None) -> dict:
    """Create a product in the catalog. `attributes` is the full category-attribute dict (all keys,
    null where unknown); `extra` is a catch-all for supplier info that fits no catalog field.
    The product is embedded and indexed, so it is searchable immediately."""
    if collection.get(ids=[sku])["ids"]:
        return {"error": f"sku {sku!r} already exists"}
    doc = name + " — " + long_description
    collection.add(
        ids=[sku],
        embeddings=[embed_model.encode(doc).tolist()],
        documents=[doc],
        metadatas=[{
            "name": name, "brand": brand, "category": category, "price": float(price),
            "short_description": short_description, "long_description": long_description,
            # Chroma metadata must be SCALAR -> store the dicts as JSON strings.
            "attributes": json.dumps(attributes or {}),
            "extra": json.dumps(extra or {}),
        }],
    )
    return {"status": "created", "sku": sku, "catalog_size": collection.count()}

_tools = await mcp_server.list_tools()
print(f"MCP server '{mcp_server.name}' ready with {len(_tools)} tools:", sorted(t.name for t in _tools))

In [ ]:
# LLM client (Haiku only) -- the agent loop runs many calls, so the key is REQUIRED for this whole TD.
load_dotenv()  # reads ANTHROPIC_API_KEY from .env at the project root
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Create a .env file at the project root with\n"
        "ANTHROPIC_API_KEY=sk-ant-...  (see resources/setup_guide.md).\n"
        "Unlike earlier labs, the ENTIRE TD5 agent loop needs the key -- it runs Haiku in a loop."
    )
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
print("Haiku client ready.")

In [ ]:
# Load the agent's SKILL (its SOP for adding a product) into a string. It goes into the system prompt.
SKILL = Path("../data/skills/add_product/SKILL.md").read_text()

# Read the supplier email and split it into per-product blurbs (tiny pre-built helper).
# The email separates products with a line of '---'; we split on that and keep the prose blurbs.
email_text = Path("../data/supplier_emails/email_01.txt").read_text()
def split_blurbs(text):
    """Split a supplier email on '---' separators; keep chunks that look like a product blurb."""
    parts = [p.strip() for p in text.split("---")]
    # A product blurb mentions a price; the header/footer chunks don't.
    return [p for p in parts if "retail" in p.lower()]

blurbs = split_blurbs(email_text)
print(f"Skill loaded: {len(SKILL)} chars. Parsed {len(blurbs)} product blurbs from the email.")
for i, b in enumerate(blurbs, 1):
    print(f"\n--- blurb {i} ---\n{b[:160]}...")

## 2. The toolbox (recall TD4) and the agent's skill (new)

**Why the catalog team cares:** the agent is only as capable as the tools it can reach and the *know-how* that tells it how to use them well. Before we build the loop, let's name the two things the agent is given.

### Tools vs. skills

A **tool** is a single action the agent can invoke: `search_products`, `get_category_attributes`, `create_product`. Tools are **what the agent *can do***. They come from your **TD4 MCP server**, discovered over the protocol and translated 1:1 into the API's `tools=` format.

A **skill** is reusable **know-how**: a named procedure (a playbook / SOP) that tells the agent **how and when** to accomplish a whole *job* by orchestrating tools. Skills are **how the agent does a job *well***. This is Anthropic's *Agent Skills* pattern — a skill is a **`SKILL.md`** (a name, a one-line description, and the instructions) that the agent loads when a task matches.

Today the agent holds **exactly one** skill — **`add_product`** — loaded into the **system prompt**. But the point is that an agent holds a *library* of skills and picks the relevant one per request, just as it picks tools by their schema. A real Fnac PIM agent would also carry `update_price`, `audit_completeness`, `translate_entry`, `merge_duplicates`… — **same tools, different skills**. (Adding a second skill is a *Going further* note at the end.)

In [ ]:
# Pre-built: translate the discovered MCP catalog into Anthropic's tools= format (the TD4 1:1 map).
async def get_anthropic_tools(session):
    listed = await session.list_tools()
    return [
        {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
        for t in listed.tools
    ]

async with connect(mcp_server._mcp_server) as session:
    anthropic_tools = await get_anthropic_tools(session)
print("Tools the agent can call:")
for t in anthropic_tools:
    print(f"  - {t['name']}: {t['description'][:70]}")

# Show the loaded skill's identity (its first lines = the YAML header: name + description).
print("\nThe agent's ONE skill (loaded into the system prompt):")
print("\n".join(SKILL.splitlines()[:4]))

**Takeaway.** The agent gets a **toolbox** (the four MCP tools — what it can do) and a **skill** (the `add_product` SOP — how to do the job well). In the loop you build next, the agent will **choose among the tools on its own**, in an order *it* decides, guided by that skill. One skill today; a real PIM agent would carry many.

## 3. Build the agent: one turn, then the loop

This is the heart of TD5 — the part that *is* the agent. We build it in **two stages**.

**Stage 1 — one turn, step by step.** A single **reason → act → observe** turn, broken into **four small cells** you fill one at a time: (1) **reason** — ask the model, giving it the tools; (2) **find the tool** it chose; (3) **act + observe** — run that tool over MCP and read the result; (4) ask again for the **final answer**. Each cell is one piece of the turn, with its own quick check — so you see each moving part on its own.

**Stage 2 — the loop (the centerpiece).** Wrap those four steps into `run_agent`, which **repeats** the turn — after each observation the model reasons again and may call another tool — until it needs no more tools and returns a final answer. A `max_iters` guard keeps it from running away.

The ingredients for the turn (the goal, the translated toolbox, the seed message) are given just below; then you build the turn one cell at a time.

In [ ]:
# Pre-built: the ingredients for ONE turn. We run the turn step by step across the next four cells
# (you fill each step), then wrap the whole thing into a loop in the centerpiece TODO.
goal = "Do we already carry waterproof bluetooth speakers? Name one if so."
# Discover + translate the toolbox once (a brief session), and seed the conversation with the goal.
async with connect(mcp_server._mcp_server) as session:
    anthropic_tools = await get_anthropic_tools(session)
messages = [{"role": "user", "content": goal}]
print("Goal:", goal)
print(f"{len(anthropic_tools)} tools available; conversation seeded with the user goal.")

In [ ]:
# TODO (turn 1/4) -- REASON. Ask the model for its first move, giving it the tool catalog.
#   Call client.messages.create(model=MODEL, max_tokens=1024, system=SKILL,
#                               tools=anthropic_tools, messages=messages) and bind it to `resp`.
raise NotImplementedError

# Self-check: the model wants to ACT (call a tool), not answer yet.
assert resp.stop_reason == "tool_use", f"expected a tool call, got stop_reason={resp.stop_reason!r}"
print("stop_reason:", resp.stop_reason, "-> the model wants to use a tool.")

In [ ]:
# TODO (turn 2/4) -- FIND THE TOOL the model chose, and PRINT it.
#   Scan resp.content for the first block whose .type == "tool_use"; bind it to `tool_use`.
#   Then print the tool name + its JSON arguments, e.g.:  -> call: search_products({...})
#   (The model wrote those arguments ITSELF, reading the tool's schema -- nobody hard-coded them.)
raise NotImplementedError

# Self-check: we found a tool_use block with a name.
assert tool_use is not None and tool_use.name, "no tool_use block found in resp.content"
print("chosen tool:", tool_use.name)

In [ ]:
# TODO (turn 3/4) -- ACT + OBSERVE. Run the chosen tool over MCP and read its result.
#   Open a session (`async with connect(mcp_server._mcp_server) as session:`) and call
#       out = await session.call_tool(tool_use.name, tool_use.input)
#   Build  result_text = "\n".join(c.text for c in out.content)  and PRINT it (truncated is fine).
raise NotImplementedError

# Self-check: the tool returned some text to observe.
assert result_text.strip(), "the tool returned no content"
print("\n(observed", len(result_text), "chars of tool output.)")

In [ ]:
# TODO (turn 4/4) -- FINAL ANSWER. Feed the observation back and ask the model AGAIN. This closes the turn.
#   1. record the assistant turn:  messages.append({"role": "assistant", "content": resp.content});
#   2. add the observation as a tool_result:
#        messages.append({"role": "user", "content": [
#            {"type": "tool_result", "tool_use_id": tool_use.id, "content": result_text}]});
#   3. call client.messages.create(... same model/max_tokens/system/tools/messages ...) -> `follow`;
#   4. answer = "".join(b.text for b in follow.content if b.type == "text").
raise NotImplementedError

# Self-check: the model produced a final natural-language answer -- ONE full turn is done.
assert isinstance(answer, str) and answer.strip(), "no final answer produced"
print("\n=== final answer (after ONE turn) ===\n" + answer)

**That's one turn — but one turn isn't an agent.** It made a single decision and stopped. A real goal (*"enrich this product"*) needs **several** tool calls in sequence: categorize, fetch the schema, search exemplars, then create. An **agent** turns the turn you just ran by hand into a **loop** — after each observation it reasons again and may call another tool — until the model stops asking for tools (`stop_reason != "tool_use"`). A `max_iters` guard stops it running away.

The shape (those four steps, now *looped* with a stop condition):

```python
messages = [{"role": "user", "content": goal}]
for _ in range(max_iters):
    resp = client.messages.create(model=MODEL, max_tokens=1024,
                                   system=SKILL, tools=anthropic_tools, messages=messages)
    if resp.stop_reason != "tool_use":      # model returned a final answer -> done
        ...return the text...
    messages.append({"role": "assistant", "content": resp.content})
    # execute EVERY tool_use block this turn, append one tool_result per block, then loop
```

In [ ]:
# TODO (the loop -- THE CENTERPIECE) -- generalize the four steps above into the reason -> act -> observe
# loop, wrapped in a function. Read this whole docstring, then write the BODY.
async def run_agent(goal, max_iters=12):
    '''Run Haiku in a tool-use loop until it stops requesting tools (the agent).

    GOAL
      Given a natural-language `goal`, let the model reason, call tools over MCP, observe the
      results, and keep going until it produces a final text answer (or the guard trips).

    STEPS (in order)
      1. Open the MCP session (`async with connect(mcp_server._mcp_server) as session:`) and build the
         tool list with `anthropic_tools = await get_anthropic_tools(session)`.
      2. Start the conversation: `messages = [{"role": "user", "content": goal}]`.
      3. Loop up to `max_iters` times. Each iteration:
         a. call `client.messages.create(model=MODEL, max_tokens=1024, system=SKILL,
            tools=anthropic_tools, messages=messages)`;
         b. if `resp.stop_reason != "tool_use"`, the model is done -> collect the text from the
            `text` blocks of `resp.content`, print a final marker, and RETURN that text;
         c. otherwise append the assistant turn: `messages.append({"role":"assistant",
            "content": resp.content})`;
         d. walk EVERY block of `resp.content`; for each block whose `.type == "tool_use"`, run the
            tool with `out = await session.call_tool(block.name, block.input)`, turn its result into
            text with `"\n".join(c.text for c in out.content)`, and collect a result dict
            `{"type": "tool_result", "tool_use_id": block.id, "content": <that text>}`;
         e. feed ALL the observations back in one user turn:
            `messages.append({"role": "user", "content": <list of tool_result dicts>})`.

    CONTRACT
      Signature: `run_agent(goal: str, max_iters: int = 12) -> str`. `goal` is a natural-language
      instruction. Returns the final assistant text (a string).

    GUARD
      A turn may contain SEVERAL tool_use blocks -- handle them all in one pass (step d), appending one
      tool_result per block. Stop after at most `max_iters` model calls; if you hit the cap without a
      final answer, return a short note saying the limit was reached (don't loop forever).

    PRINT (the visible trace -- this is what makes the loop legible)
      Each time the agent calls a tool, print one line like:  `-> call: search_products({...})`
      (the tool name + its JSON arguments). This trace is the whole point -- you SEE the agent reason,
      act, and observe, step by step.
    '''
    raise NotImplementedError

# Self-check: run the agent on a simple Q&A goal and confirm it answered (watch the trace above).
answer = await run_agent("Do we already carry waterproof bluetooth speakers? Name one if so.")
assert isinstance(answer, str) and answer.strip(), "agent returned no final answer"
print("\nLoop self-check: got a non-empty answer of", len(answer), "chars.")
print("NOTE: confirm the trace above shows a '-> call: search_products(...)' line.")

**Takeaway.** You turned your single **reason → act → observe** turn into an **autonomous loop**: the same turn, repeated, with the model choosing each tool call and stopping itself when done. **That loop *is* the agent.** Everything below is just giving it a harder goal.

## 4. Enrich the supplier products — the payoff

Now the real job: turn each messy blurb in the supplier email into a **complete, on-brand catalog entry** and create it. Here's the key design point — **the agent already knows *how***. Its `add_product` **skill** (loaded into the system prompt) holds the whole SOP: pick the leaf category, fetch its attributes, search similar products for the voice, fill **every** attribute (`null` where unknown), route leftovers to `extra`, use the **retail** price, then `create_product`.

So the **goal we hand the agent is deliberately thin** — it just gives it *one blurb* and says "add this, following your skill." That's the whole **tool-vs-skill lesson in action**: the **skill is the reusable know-how; the goal is only the task + the data.** If we re-typed the rules into every goal, the skill would be pointless. Both `enrichment_goal` and the `enrich_all` driver below are **given** — driving the agent over the whole email is just a thin `for` loop, because the real work was the loop you wrote in §3. Run it and watch.

In [ ]:
# Pre-built: the goal is THIN on purpose. The add_product SKILL (in the system prompt) already holds the full
# SOP -- pick category, fetch attributes, search exemplars, fill null/extra, use the retail price, create. So
# the goal only hands the agent ONE blurb and tells it to follow that skill. Re-typing the rules here would
# defeat the whole point of having a skill.
def enrichment_goal(blurb):
    """Wrap ONE supplier blurb into a natural-language goal for run_agent."""
    return (
        "Add this new supplier product to our catalog, following your add_product skill. "
        "When done, report the SKU you created and a one-line summary.\n\n"
        f"Supplier blurb:\n{blurb}"
    )

print("Example thin goal:\n", enrichment_goal(blurbs[0])[:220], "...")

> **Predict (10 seconds — just for you).** Which tools will the agent call, and in what order? Will it really fill *every* category attribute, even the ones the blurb never mentions? Jot a guess, then run the next cell and watch the trace.

In [ ]:
# Pre-built: the thin driver -- ONE agent run per blurb. The real work was run_agent (§3); this just loops
# over the email, collecting each product's final summary. run_agent is async, so we await it per blurb.
async def enrich_all(blurbs):
    '''Run the enrichment agent on every product blurb in the supplier email.'''
    answers = []
    for i, blurb in enumerate(blurbs, 1):
        print(f"\n{'='*70}\nENRICHING PRODUCT {i}\n{'='*70}")
        answer = await run_agent(enrichment_goal(blurb))
        answers.append(answer)
        print("\n--- agent summary ---\n" + answer)
    return answers

# Run the agent over the whole email. Watch each product's reason->act->observe trace.
results = await enrich_all(blurbs)
print(f"\nEnriched {len(results)} products.")

**Takeaway.** With nothing but a messy prose blurb and its skill, the agent **categorized** the product, **fetched the schema**, **grounded** itself in similar catalog entries, **generated** an on-brand entry, and **created** the product — deciding each tool call *itself*. You wrote the loop and the goal; the agent did the job.

## 5. Inspect the agent's work (the freshness finale)

Did the agent obey the rules? Let's fetch the products it created and check three things: the descriptions are **on-brand**, **every** category attribute is present (`null` where the blurb was silent), and the leftover supplier details landed in **`extra`**. Then we search for the new products — they're retrievable seconds after creation, with no reindexing.

In [ ]:
# Find the SKUs the agent created (anything not in the original catalog) and fetch each one back.
original_skus = set(df["sku"])
new_skus = [sku for sku in collection.get()["ids"] if sku not in original_skus]
print(f"Agent created {len(new_skus)} new products: {new_skus}\n")

async with connect(mcp_server._mcp_server) as session:
    for sku in new_skus:
        res = await session.call_tool("get_product", {"sku": sku})
        p = json.loads(res.content[0].text)
        print(f"{'='*70}")
        print(f"{p['name']}  ({p['category']}, €{p['price']:.0f})  [{sku}]")
        print(f"  short: {p['short_description']}")
        print(f"  long : {p['long_description']}")
        print(f"  category attributes: {json.dumps(p['attributes'])}")
        print(f"  extra              : {json.dumps(p['extra'])}")

In [ ]:
# Freshness: a product is searchable the INSTANT the agent creates it -- no reindex, no restart. Search the
# catalog for each product the agent just added and confirm it now comes back among the hits.
async with connect(mcp_server._mcp_server) as session:
    for sku in new_skus:
        prod = json.loads((await session.call_tool("get_product", {"sku": sku})).content[0].text)
        res = await session.call_tool("search_products", {"query": prod["name"], "k": 3})
        hits = [json.loads(c.text) for c in res.content]
        found = any(h["sku"] == sku for h in hits)
        print(f"search {prod['name']!r} -> {'FOUND (created moments ago)' if found else 'NOT FOUND'}")
        for h in hits:
            mark = "   <- the one the agent just created" if h["sku"] == sku else ""
            print(f"    - {h['name']}  ({h['category']}, €{h['price']:.0f}){mark}")
        assert found, f"{sku} should be retrievable immediately after creation"
print("\nFreshness confirmed: new products are queryable seconds after the agent writes them -- no reindex.")

In [ ]:
# Check (the §3-spec rule, surfaced not asserted): every created product should carry EVERY
# category-attribute key (null allowed) and an `extra` field. The agent is nondeterministic, so we
# REPORT any slip instead of crashing the notebook on an otherwise-clean run.
async with connect(mcp_server._mcp_server) as session:
    for sku in new_skus:
        pres = await session.call_tool("get_product", {"sku": sku})
        p = json.loads(pres.content[0].text)
        ares = await session.call_tool("get_category_attributes", {"category": p["category"]})
        schema = json.loads(ares.content[0].text)
        missing = set(schema) - set(p["attributes"])
        if missing:
            print(f"{sku}: ⚠️ agent left category keys unset: {missing} "
                  f"(re-run; the add_product skill says include every key, null where unknown)")
            continue
        assert "extra" in p, f"{sku} has no extra field"
        print(f"{sku}: all {len(schema)} category attributes present; extra has {len(p['extra'])} entries. OK")
print("\nSelf-check passed: every product is complete (null where unknown) and carries an extra payload.")

**Takeaway — the whole thread closes.** **TD1** embedded the catalog into a vector space; **TD3** searched it with RAG; **TD4** exposed the search and the schema as MCP tools; **TD5's loop** orchestrated all of them to turn a messy email into complete, on-brand, immediately-searchable catalog entries. The promise of TD1 — *similar products are close, so we can search and reuse them* — literally paid off in the agent you just built.

## Reflection 1 — autonomy vs. control

Your loop has a `max_iters` guard, and the agent decides its own tool calls. **What can go wrong without the guard?** And more broadly: what's the trade-off between letting the agent decide freely vs. constraining it? Give **one** concrete failure mode you'd guard against in production (interpret *your* run — what did the agent actually do?).

**Your answer** — write it yourself. You may use AI to help you think, but the words must be yours.

_..._

## Reflection 2 — the `null` + `extra` design

Why force the agent to return **every** category attribute (even as `null`) and route leftovers into **`extra`**, rather than just keeping whatever it happened to find? What does this design buy the PIM **downstream**?

**Your answer** — write it yourself.

_..._

## 6. Wrap-up + your mini-project

**The transferable skill.** An **AI agent** = an LLM **+ tools** (MCP) **+ a skill/SOP** (the `SKILL.md` in the system prompt) **+ a reason → act → observe loop** **+ a goal**. Give it those, and it composes the steps itself. You built that loop from scratch — no framework — so you can see exactly what an agent *is*.

**Where your bricks returned:** **TD2**'s categorization is now the agent's own reasoning over the tree; **TD3**'s RAG is the `search_products` tool that grounds the on-brand writing; **TD4**'s MCP server is the agent's whole toolbox; **TD5** is the loop that drives them. This is the enterprise agent from the lecture's final slide — rebuilt from your own code.

**Bridge to the hackathon.** This toolkit — RAG, MCP, an agent loop — is your starter kit. Your hackathon build must reuse **at least one** of them (ideally combined).

---

### Mini-project brief — the **PIM Copilot** *(build it yourself)*

*(Markdown only — no code here. You build this **from scratch** with Claude Code; we ship no scaffold. The whole point is that you *can* ship a real web-app this way.)*

Turn the loop into a small **chat web-app a Fnac catalog manager would actually use** — *your own Claude Desktop*, but for your PIM. Put **all** the code in a **`mini_project/`** folder next to this notebook (`notebooks/TD5_agent/mini_project/`). The **end goal**: paste a messy supplier blurb into your chatbot and **watch the new product appear in the PIM**, fully attributed — your agent enriching a real catalog.

**Three pieces:**

1. **Python backend = the agent.** Your `run_agent` reason→act→observe loop + Haiku + an **MCP stdio client** to **your TD4 mini-project server** (out-of-process — this reinforces TD4; the agent spawns the server and talks to it over stdio, not in-memory), carrying the `add_product` **skill** in the system prompt. Expose **one HTTP endpoint** — `POST /chat` — that takes the conversation, runs the loop, and returns the assistant reply **plus the tool-call trace**. (**FastAPI or Flask** — your choice.) Point that server at a **persistent** `chroma_index` folder on disk (the TD4 solution already does) — that shared on-disk index is what makes piece 3 work. **Heads-up:** the `add_product` skill fills the **full `attributes` dict** (every category key, `null` where unknown) plus an **`extra`** catch-all for supplier leftovers, so your TD4 `create_product` must accept **both** — TD4 left that tool's exact fields to you, so extend it if your version didn't.
2. **JS frontend = the chat UI.** **Vue.js recommended** (any JS framework is fine — you don't know web dev, but Claude Code makes this very doable, and *that's part of the lesson*). It renders the conversation and **shows the tool calls as they happen** — the reason→act→observe trace, live.
3. **See your products land in the PIM.** A small **read-only PIM visualizer** is **provided for you** at [`notebooks/pim-prod/`](../pim-prod/) — *you don't build this one; just run it per its README and point it at your index*. It's a FastAPI + Vue app that browses any ChromaDB `chroma_index` and shows every product's attributes, **completeness**, and nearest neighbours. **Point it at the SAME `chroma_index` your TD4 stdio server writes to** (use its **Index** picker, or start it with `PIM_INDEX_DIR=…`). Then the payoff loop: paste a supplier blurb in your copilot → the agent `create_product`s → **refresh the PIM and watch it show up**, fully attributed (`null`s flagged, `extra` preserved). That round-trip — *your chatbot writes, the PIM shows it* — is the whole point.

**What the manager can do:**
- **Ask the catalog** — "what ANC headphones under €300 do we carry?" → the agent calls `search_products` and answers;
- **Enrich/add** — paste a **messy supplier blurb** → the agent categorizes, fills attributes (`null` + `extra`), and `create_product`s → the product is **instantly searchable** and **appears in the PIM**.

**Recommended UX (full version = stretch):** *human-in-the-loop* — the agent **drafts** the entry and the manager **confirms** before the write commits. Simple version: the agent creates and the UI shows the trace; the propose→confirm flow is *going further*.

**Hygiene:** no API key in code (`.env`); a `requirements.txt` (backend) + minimal frontend deps; a short `README.md` with run instructions — how to start the TD4 stdio server / point the client at it, run the backend, run the frontend, and open the PIM visualizer on the same index. It **reuses your TD4 mini-project server** as its tool source.

### 🚀 Going further (optional — no solutions; use Claude Code)
- **A second skill.** Add `update_price` or `audit_completeness` as another `SKILL.md`, build a tiny skill registry, and let the agent **pick the skill** by its description before acting — the *"an agent has many skills"* idea, made hands-on.
- **A trickier email.** Add a second supplier email with an **ambiguous** product (a category the agent has to reason hard about) and watch how it decides.
- **Swap the loop for a framework** (**LangGraph** / **Google ADK**) and compare — what does the framework add, and what does it hide from you?
- **A pure Q&A goal.** Give the agent a retrieve-to-answer goal over the catalog — same loop, different goal. ✅

## 🧰 Why an agent framework — and why this TD doesn't use one

You wrote `run_agent` in ~30 lines, and that loop *is* a complete agent engine. Before reaching for **LangGraph / Google ADK / CrewAI / AutoGen / the OpenAI Agents SDK**, it pays to know precisely what such a framework would do *for* you — and what it would take *away*. (This is the long answer to the *Going further* bullet above.)

### First: what your hand-rolled loop already does

Look back at `run_agent`. With no framework, it already owns **every** moving part of an agent:

- the **reason → act → observe loop** and its stop condition (`stop_reason != "tool_use"`);
- the **`max_iters` runaway guard** (an agent that never stops is a real failure mode — see Reflection 1);
- the **message bookkeeping** — append the assistant turn, then **one `tool_result` per `tool_use` block**, in order (a classic source of bugs);
- the **tool dispatch** — receive a `tool_use`, run `session.call_tool(name, input)`, reformat the result into a `tool_result`;
- the **MCP → `tools=` schema map** (the 1:1 translation from TD4);
- the **trace** (`-> call: search_products({...})`) that makes the agent legible.

A framework hands you all of the above pre-built. The real question is the trade-off.

### What a framework *adds* (the upside)

- **State & memory, including persistence.** Conversation history, short/long-term memory, and **checkpointing**: kill the process and *resume the agent mid-task* where it left off. (Your mini-project persists Chroma to disk; a framework persists the **agent's own state** too.)
- **Richer control flow.** Your loop only knows *"one more tool, or stop."* A framework models a **graph / state machine**: conditional branches (*"if the category is ambiguous, ask a human"*), loops back to earlier steps, **parallel** tool calls, and **multi-agent** orchestration (a supervisor delegating to specialist agents).
- **Human-in-the-loop, built in.** Pause before a write, wait for approval, then resume. That is *exactly* the **propose → confirm** flow listed as a stretch goal for your PIM Copilot — LangGraph offers it natively via *interrupts*.
- **Production plumbing.** Retries, rate-limit backoff, timeouts, streaming, and **structured tracing**: your `print("-> call: …")` becomes a filterable, replayable trace with per-step token costs (e.g. LangSmith).
- **Provider abstraction.** Swap models or providers behind one interface, instead of the hard-coded `MODEL = "claude-haiku-4-5"`.

### What a framework *hides* (the downside)

- **Transparency.** The loop becomes a black box. You just built one by hand *so you'd know what an agent is* — a framework re-hides it. When it misbehaves, you now have to debug **your code *and* the framework's internals**.
- **Implicit prompts & cost.** Many frameworks inject their own prompts, parsing, and retries that you never see. That can quietly change output quality, **inflate token usage**, or defeat prompt caching — and TD5 is already your most API-intensive lab.
- **Over-engineering.** For a 30-line loop, importing a large framework adds dependencies and new concepts (nodes, typed state, edges) whose learning curve can exceed the problem you actually have.
- **Churn & lock-in.** This ecosystem moves fast and breaks its APIs often. You end up learning a framework's abstractions — coupled to its release cycle — rather than the model underneath.

### Rule of thumb

**Code the loop once by hand (you just did) so you understand the moving parts. Reach for a framework when a *real* need appears — durable state, branching, human approval, or several cooperating agents — not before.** For the PIM Copilot, your own `run_agent` is plenty; a framework starts earning its keep the day you add propose→confirm or a second cooperating agent.